# 02 ? Data Quality & Validation

## 1. Introduction and objectives

Phase 1A helped us understand what Olist contains; Phase 1B checks whether its values and relationships are sufficiently consistent for later analysis. Exploration describes the data, while validation tests expectations and reports exceptions without correcting them. We validate orders, order items, customers, payments, reviews, products, and sellers when available. They provide order progress and commerce context for CallVerse scenarios involving tracking, delays, returns, refunds, and cancellations. Olist does not record actual customer contacts.

## 2. Imports and dataset loading

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "olist"
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR, "exists:", DATA_DIR.exists())
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset directory missing: {DATA_DIR}. Put Olist CSVs in data/raw/olist/.")

selected_files = {
    "orders": "olist_orders_dataset.csv", "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv", "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv", "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
}
datasets = {}
for dataset_name, file_name in selected_files.items():
    file_path = DATA_DIR / file_name
    if file_path.exists():
        datasets[dataset_name] = pd.read_csv(file_path)
    else:
        print("Dataset unavailable; related checks will be skipped:", file_path)
orders_df = datasets.get("orders")
quality_findings = []

Project root: c:\Users\user\OneDrive - ESPRIT\Bureau\CallVerse
Data directory: c:\Users\user\OneDrive - ESPRIT\Bureau\CallVerse\data\raw\olist exists: True


## 3. Dataset overview

This inventory records dimensions, columns, pandas-inferred types, and memory use. Date columns are parsed explicitly later.

In [2]:
overview_rows = []
for dataset_name, dataset in datasets.items():
    overview_rows.append({"dataset": dataset_name, "rows": len(dataset), "columns": len(dataset.columns),
        "column_names": ", ".join(dataset.columns.astype(str)),
        "data_types": "; ".join(f"{column}: {dtype}" for column, dtype in dataset.dtypes.items()),
        "memory_mb": round(dataset.memory_usage(index=True, deep=True).sum() / 1024**2, 2)})
display(pd.DataFrame(overview_rows))

,dataset,rows,columns,column_names,data_types,memory_mb
0,orders,99441,8,"order_id, customer_id, order_status, order_pur...",order_id: object; customer_id: object; order_s...,52.94
1,order_items,112650,7,"order_id, order_item_id, product_id, seller_id...",order_id: object; order_item_id: int64; produc...,35.99
2,customers,99441,5,"customer_id, customer_unique_id, customer_zip_...",customer_id: object; customer_unique_id: objec...,26.59
3,payments,103886,5,"order_id, payment_sequential, payment_type, pa...",order_id: object; payment_sequential: int64; p...,16.23
4,reviews,99224,7,"review_id, order_id, review_score, review_comm...",review_id: object; order_id: object; review_sc...,39.12
5,products,32951,9,"product_id, product_category_name, product_nam...",product_id: object; product_category_name: obj...,6.30
6,sellers,3095,4,"seller_id, seller_zip_code_prefix, seller_city...",seller_id: object; seller_zip_code_prefix: int...,0.59


## 4. Missing values analysis

For every column, count missing values and their percentage of rows, sorted by percentage. Missing delivery times may be legitimate for canceled or in-progress orders. This phase reports missingness without filling values.

In [3]:
callverse_columns = {"orders": ["order_status", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"], "order_items": ["order_id", "product_id", "seller_id"], "customers": ["customer_id"], "payments": ["order_id", "payment_type", "payment_value"], "reviews": ["order_id", "review_score"], "products": ["product_id"], "sellers": ["seller_id"]}
missing_summaries = {}
for dataset_name, dataset in datasets.items():
    missing_count = dataset.isna().sum()
    summary = pd.DataFrame({"column": missing_count.index, "missing_values": missing_count.values,
        "missing_percent": missing_count.values / len(dataset) * 100 if len(dataset) else 0})
    summary = summary.sort_values("missing_percent", ascending=False)
    missing_summaries[dataset_name] = summary
    for _, row in summary[summary.missing_values > 0].iterrows():
        quality_findings.append({"Check": "Missing values", "Dataset": dataset_name, "Issue investigated": row.column, "Number of affected records": int(row.missing_values), "Interpretation": f"{row.missing_percent:.2f}% missing; assess in context."})
    print("\nMissing values:", dataset_name)
    display(summary)
    relevant = summary[(summary.missing_values > 0) & summary.column.isin(callverse_columns.get(dataset_name, []))]
    if not relevant.empty:
        print("CallVerse-relevant missing columns:")
        display(relevant)


Missing values: orders


,column,missing_values,missing_percent
6,order_delivered_customer_date,2965,2.981668
5,order_delivered_carrier_date,1783,1.793023
4,order_approved_at,160,0.160899
0,order_id,0,0.000000
1,customer_id,0,0.000000
2,order_status,0,0.000000
3,order_purchase_timestamp,0,0.000000
7,order_estimated_delivery_date,0,0.000000


CallVerse-relevant missing columns:


,column,missing_values,missing_percent
6,order_delivered_customer_date,2965,2.981668
5,order_delivered_carrier_date,1783,1.793023
4,order_approved_at,160,0.160899



Missing values: order_items


...

,column,missing_values,missing_percent
3,review_comment_title,87656,88.341530
4,review_comment_message,58247,58.702532
0,review_id,0,0.000000
1,order_id,0,0.000000
2,review_score,0,0.000000
5,review_creation_date,0,0.000000
6,review_answer_timestamp,0,0.000000



Missing values: products


,column,missing_values,missing_percent
1,product_category_name,610,1.851234
2,product_name_lenght,610,1.851234
3,product_description_lenght,610,1.851234
4,product_photos_qty,610,1.851234
5,product_weight_g,2,0.006070
6,product_length_cm,2,0.006070
7,product_height_cm,2,0.006070
8,product_width_cm,2,0.006070
0,product_id,0,0.000000



Missing values: sellers


,column,missing_values,missing_percent
0,seller_id,0,0.0
1,seller_zip_code_prefix,0,0.0
2,seller_city,0,0.0
3,seller_state,0,0.0


## 5. Duplicate analysis

Completely identical rows differ from repeated identifiers. Check expected record keys only: order and customer IDs, product and seller IDs, review IDs, and the order plus item number composite. Order IDs can repeat in items, payments, and reviews; customer IDs can repeat across orders. No rows are deleted.

In [4]:
duplicate_findings = []
key_expectations = {"orders": ["order_id"], "customers": ["customer_id"], "order_items": [["order_id", "order_item_id"]], "products": ["product_id"], "sellers": ["seller_id"], "reviews": ["review_id"]}
for name, dataset in datasets.items():
    exact_count = int(dataset.duplicated().sum())
    duplicate_findings.append({"dataset": name, "check": "Completely duplicated rows", "columns": "all columns", "affected_records": exact_count})
    quality_findings.append({"Check": "Duplicates", "Dataset": name, "Issue investigated": "Completely duplicated rows", "Number of affected records": exact_count, "Interpretation": "Exact duplicate count; rows retained."})
    for key in key_expectations.get(name, []):
        columns = key if isinstance(key, list) else [key]
        if all(column in dataset for column in columns):
            repeated_count = int(dataset.duplicated(subset=columns, keep=False).sum())
            duplicate_findings.append({"dataset": name, "check": "Repeated expected identifier", "columns": ", ".join(columns), "affected_records": repeated_count})
            quality_findings.append({"Check": "Identifier uniqueness", "Dataset": name, "Issue investigated": ", ".join(columns), "Number of affected records": repeated_count, "Interpretation": "Rows with repeated expected record key; inspect before joining."})
    for column in ["order_id", "customer_id"]:
        if column in dataset and column not in key_expectations.get(name, []):
            count = int(dataset[column].duplicated(keep=False).sum())
            if count:
                duplicate_findings.append({"dataset": name, "check": "Repeated identifier (may be legitimate)", "columns": column, "affected_records": count})
duplicate_summary = pd.DataFrame(duplicate_findings)
display(duplicate_summary)

,dataset,check,columns,affected_records
0,orders,Completely duplicated rows,all columns,0
1,orders,Repeated expected identifier,order_id,0
2,order_items,Completely duplicated rows,all columns,0
3,order_items,Repeated expected identifier,order_id,23787
4,order_items,Repeated expected identifier,order_item_id,112649
5,customers,Completely duplicated rows,all columns,0
6,customers,Repeated expected identifier,customer_id,0
7,payments,Completely duplicated rows,all columns,0
8,payments,Repeated identifier (may be legitimate),order_id,7407
9,reviews,Completely duplicated rows,all columns,0


## 6. Identifier and referential integrity validation

Compare each available child identifier with the parent identifier set. Repeated child keys are allowed. Missing keys are counted separately from unmatched non-missing references.

In [5]:
relationship_specs = [("Orders to customers", "orders", "customer_id", "customers", "customer_id"), ("Items to orders", "order_items", "order_id", "orders", "order_id"), ("Items to products", "order_items", "product_id", "products", "product_id"), ("Items to sellers", "order_items", "seller_id", "sellers", "seller_id"), ("Payments to orders", "payments", "order_id", "orders", "order_id"), ("Reviews to orders", "reviews", "order_id", "orders", "order_id")]
relationship_results = []
for label, child_name, child_key, parent_name, parent_key in relationship_specs:
    child, parent = datasets.get(child_name), datasets.get(parent_name)
    if child is None or parent is None or child_key not in child or parent_key not in parent:
        relationship_results.append({"relationship": label, "status": "table or key unavailable"})
        continue
    values = child[child_key]
    nonmissing = values.dropna()
    unmatched = ~nonmissing.isin(set(parent[parent_key].dropna().unique()))
    count = int(unmatched.sum())
    relationship_results.append({"relationship": label, "status": "checked", "child_rows": len(child), "distinct_child_ids": nonmissing.nunique(), "distinct_parent_ids": parent[parent_key].nunique(), "missing_child_ids": int(values.isna().sum()), "unmatched_references": count})
    quality_findings.append({"Check": "Referential integrity", "Dataset": child_name, "Issue investigated": label, "Number of affected records": count, "Interpretation": "Unmatched non-missing references; review if non-zero."})
display(pd.DataFrame(relationship_results))

,relationship,status,child_rows,distinct_child_ids,distinct_parent_ids,missing_child_ids,unmatched_references
0,Orders to customers,checked,99441,99441,99441,0,0
1,Items to orders,checked,112650,98666,99441,0,0
2,Items to products,checked,112650,32951,32951,0,0
3,Items to sellers,checked,112650,3095,3095,0,0
4,Payments to orders,checked,103886,99440,99441,0,0
5,Reviews to orders,checked,99224,98673,99441,0,0


## 7. Timestamp consistency validation

Parse the available order milestones, count non-empty values that fail parsing, and compare chronology only when both values exist. Suspicious order is worth investigation but is not proof of error. Missing events may match an order's state. Later predictive features must use only information available at simulated contact time, not future delivery outcomes.

In [6]:
timestamp_columns = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]
timestamp_results = []
if orders_df is not None:
    for column in timestamp_columns:
        if column in orders_df:
            parsed = column + "_parsed"
            orders_df[parsed] = pd.to_datetime(orders_df[column], errors="coerce")
            invalid = int((orders_df[column].notna() & orders_df[parsed].isna()).sum())
            timestamp_results.append({"check": "Unparseable non-missing timestamps", "comparison": column, "comparable_records": int(orders_df[column].notna().sum()), "suspicious_records": invalid})
            quality_findings.append({"Check": "Timestamp parsing", "Dataset": "orders", "Issue investigated": column, "Number of affected records": invalid, "Interpretation": "Non-empty values that did not parse."})
    pairs = [("order_purchase_timestamp", "order_approved_at"), ("order_approved_at", "order_delivered_carrier_date"), ("order_delivered_carrier_date", "order_delivered_customer_date"), ("order_purchase_timestamp", "order_delivered_customer_date")]
    for earlier, later in pairs:
        a, b = earlier + "_parsed", later + "_parsed"
        if a in orders_df and b in orders_df:
            both = orders_df[a].notna() & orders_df[b].notna()
            count = int((both & (orders_df[a] > orders_df[b])).sum())
            label = earlier + " after " + later
            timestamp_results.append({"check": "Chronological order", "comparison": label, "comparable_records": int(both.sum()), "suspicious_records": count})
            quality_findings.append({"Check": "Timestamp consistency", "Dataset": "orders", "Issue investigated": label, "Number of affected records": count, "Interpretation": "Compared where both times exist; investigate, not a confirmed error."})
    actual, estimate, purchase = "order_delivered_customer_date_parsed", "order_estimated_delivery_date_parsed", "order_purchase_timestamp_parsed"
    if actual in orders_df and estimate in orders_df:
        both = orders_df[actual].notna() & orders_df[estimate].notna()
        cases = [("Negative purchase-to-delivery duration", both & orders_df[purchase].notna() & (orders_df[actual] < orders_df[purchase])), ("Actual delivery later than estimate", both & (orders_df[actual] > orders_df[estimate]))]
        for label, mask in cases:
            count = int(mask.sum())
            timestamp_results.append({"check": "Delivery comparison", "comparison": label, "comparable_records": int(both.sum()), "suspicious_records": count})
            quality_findings.append({"Check": "Timestamp consistency", "Dataset": "orders", "Issue investigated": label, "Number of affected records": count, "Interpretation": "Delivery after estimate indicates lateness, not necessarily erroneous data."})
display(pd.DataFrame(timestamp_results))

,check,comparison,comparable_records,suspicious_records
0,Unparseable non-missing timestamps,order_purchase_timestamp,99441,0
1,Unparseable non-missing timestamps,order_approved_at,99281,0
2,Unparseable non-missing timestamps,order_delivered_carrier_date,97658,0
3,Unparseable non-missing timestamps,order_delivered_customer_date,96476,0
4,Unparseable non-missing timestamps,order_estimated_delivery_date,99441,0
5,Chronological order,order_purchase_timestamp after order_approved_at,99281,0
6,Chronological order,order_approved_at after order_delivered_carrie...,97644,1359
7,Chronological order,order_delivered_carrier_date after order_deliv...,96475,23
8,Chronological order,order_purchase_timestamp after order_delivered...,96476,0
9,Delivery comparison,Negative purchase-to-delivery duration,96476,0


## 8. Order status and business consistency

Show statuses actually present, counts, and percentages. Delivery timestamp patterns are diagnostic: missing dates can be legitimate, and unusual combinations should be investigated rather than treated as absolute violations.

In [7]:
if orders_df is not None and "order_status" in orders_df:
    counts = orders_df.order_status.value_counts(dropna=False)
    display(pd.DataFrame({"orders": counts, "percent": counts / len(orders_df) * 100 if len(orders_df) else 0}))
    print("Statuses observed:", orders_df.order_status.dropna().unique().tolist())
    delivery_columns = [column for column in ["order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"] if column in orders_df]
    display(orders_df.groupby("order_status", dropna=False)[delivery_columns].agg(lambda values: int(values.notna().sum())))
    if "delivered" in orders_df.order_status.dropna().unique() and "order_delivered_customer_date" in orders_df:
        count = int((orders_df.order_status.eq("delivered") & orders_df.order_delivered_customer_date.isna()).sum())
        quality_findings.append({"Check": "Status and delivery timestamps", "Dataset": "orders", "Issue investigated": "Delivered without customer delivery timestamp", "Number of affected records": count, "Interpretation": "Diagnostic count, interpret with other order fields."})
    if "canceled" in orders_df.order_status.dropna().unique() and "order_delivered_customer_date" in orders_df:
        count = int((orders_df.order_status.eq("canceled") & orders_df.order_delivered_customer_date.notna()).sum())
        quality_findings.append({"Check": "Status and delivery timestamps", "Dataset": "orders", "Issue investigated": "Canceled with customer delivery timestamp", "Number of affected records": count, "Interpretation": "Diagnostic count; not by itself proof of error."})
else:
    print("Order status checks unavailable: orders or order_status missing.")

,orders,percent
order_status,,
delivered,96478,97.020344
shipped,1107,1.113223
canceled,625,0.628513
unavailable,609,0.612423
invoiced,314,0.315765
processing,301,0.302692
created,5,0.005028
approved,2,0.002011


Statuses observed: ['delivered', 'invoiced', 'shipped', 'processing', 'unavailable', 'canceled', 'created', 'approved']


,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,
approved,0,0,2
canceled,75,6,625
created,0,0,5
delivered,96476,96470,96478
invoiced,0,0,314
processing,0,0,301
shipped,1107,0,1107
unavailable,0,0,609


## 9. Cross-table consistency checks

Compare distinct order coverage and count orders without item or payment rows. Different tables have different granularity and lifecycle coverage; unusual counts are observations requiring investigation.

In [8]:
coverage = []
if orders_df is not None and "order_id" in orders_df:
    order_ids = set(orders_df.order_id.dropna().unique())
    coverage.append({"dataset": "orders", "rows": len(orders_df), "distinct order IDs": len(order_ids)})
    for name in ["order_items", "payments", "reviews"]:
        dataset = datasets.get(name)
        if dataset is not None and "order_id" in dataset:
            child_ids = set(dataset.order_id.dropna().unique())
            absent = len(order_ids - child_ids)
            coverage.append({"dataset": name, "rows": len(dataset), "distinct order IDs": len(child_ids), "orders without matching rows": absent})
            if name in ["order_items", "payments"]:
                quality_findings.append({"Check": "Order coverage", "Dataset": "orders and " + name, "Issue investigated": "Orders without associated " + name, "Number of affected records": absent, "Interpretation": "Coverage observation; investigate table scope before calling it a defect."})
display(pd.DataFrame(coverage))

,dataset,rows,distinct order IDs,orders without matching rows
0,orders,99441,99441,NaN
1,order_items,112650,98666,775.0
2,payments,103886,99440,1.0
3,reviews,99224,98673,768.0


## 10. Data quality summary

The consolidated table uses counts calculated above and has no predetermined expected counts. Detailed missing-value and duplicate summaries remain in their earlier sections.

In [9]:
quality_summary = pd.DataFrame(quality_findings, columns=["Check", "Dataset", "Issue investigated", "Number of affected records", "Interpretation"])
display(quality_summary)

,Check,Dataset,Issue investigated,Number of affected records,Interpretation
0,Referential integrity,orders,Orders to customers,0,Unmatched non-missing references; review if no...
1,Referential integrity,order_items,Items to orders,0,Unmatched non-missing references; review if no...
2,Referential integrity,order_items,Items to products,0,Unmatched non-missing references; review if no...
3,Referential integrity,order_items,Items to sellers,0,Unmatched non-missing references; review if no...
4,Referential integrity,payments,Payments to orders,0,Unmatched non-missing references; review if no...
5,Referential integrity,reviews,Reviews to orders,0,Unmatched non-missing references; review if no...
6,Timestamp parsing,orders,order_purchase_timestamp,0,Non-empty values that did not parse.
7,Timestamp parsing,orders,order_approved_at,0,Non-empty values that did not parse.
8,Timestamp parsing,orders,order_delivered_carrier_date,0,Non-empty values that did not parse.
9,Timestamp parsing,orders,order_delivered_customer_date,0,Non-empty values that did not parse.


## 11. CallVerse implications

Order, customer, and product identifiers and order milestones can describe observed commerce behavior, subject to missingness and join checks. Approval, handoff, actual delivery, and estimated delivery timestamps need careful treatment; delay features should use comparable timestamps. Validate keys before joins and remember that repeated order IDs in items or payments reflect table granularity.

Olist is an e-commerce dataset, not a call-center dataset. It contains no actual customer contact events, agent assignments, call durations, queue waiting times, or abandonment records. Contact-demand modeling therefore requires transparent assumptions or a contact-generation mechanism. A delay or cancellation does not prove that a customer contacted support.

## 12. Conclusion and next steps

Phase 1B identifies missingness, duplicate patterns, identifier relationships, timestamp consistency, and order-status observations. It does not perform final cleaning or imputation. The next phase is **Phase 1C ? Data Cleaning and Preparation**, with decisions based on the findings documented here.